# Sesión 4 · Preprocesamiento de datos - SOLUCIÓN COMPLETA
**Curso:** Uso de IA para el análisis descriptivo y predictivo en energías renovables

**Objetivo:** Entender cada método de limpieza, paso a paso.

**Estructura:** Una celda = Un concepto

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
pd.set_option('display.max_columns', 20)
print('✓ Librerías cargadas')

## PARTE 1: CARGAR Y PREPARAR DATOS
Primero cargamos los datos reales desde GitHub

In [ ]:
url = 'https://raw.githubusercontent.com/anclopezg/curso-analisis-datos-energias-renovables/main/docs/datasets/datos_estacion.csv'
df = pd.read_csv(url)
print(f'✓ Dataset cargado: {df.shape[0]} filas × {df.shape[1]} columnas')
print(df.head())

In [ ]:
df_limpio = df.copy()
df = df.copy()
rng = np.random.default_rng(42)
print('✓ Backup creado')

In [ ]:
# Introducir problemas
df.loc[rng.choice(len(df), 50, replace=False), 'irradiancia_wm2'] = np.nan
df.loc[rng.choice(len(df), 30, replace=False), 'temperatura_c'] = -999
df.loc[rng.choice(len(df), 15, replace=False), 'velocidad_viento_kmh'] *= 10
df = pd.concat([df, df.iloc[100:120]], ignore_index=True)
print('✓ Problemas introducidos')

## PARTE 2: IMPUTACIÓN - MÉTODOS DE RELLENAR FALTANTES

¿Qué es la imputación?
- Rellenar valores faltantes (NaN) con estimaciones
- Basadas en los datos disponibles
- Cada método asume algo diferente sobre los datos

### MÉTODO 1: ELIMINAR (dropna)

**Cómo funciona:** Simplemente elimina filas con NaN

**Fórmula:** Si hay NaN → borra la fila

**Cuándo usar:**
- Datos con <5% faltantes
- No es crítico perder esas observaciones
- Datos NO temporal

**Ventajas:**
- Simple
- Sin suposiciones

**Desventajas:**
- ❌ Pierdes datos
- ❌ Rompe continuidad temporal

**Para datos espaciales:** ❌ No recomendado (pierdes información)

In [ ]:
col_demo = 'irradiancia_wm2'
valores_con_nan = df[col_demo].copy()

print('ANTES:')
print(f'  Total de datos: {len(valores_con_nan)}')
print(f'  Faltantes: {valores_con_nan.isna().sum()}')

# APLICAR MÉTODO 1: ELIMINAR
resultado_eliminar = valores_con_nan.dropna()

print('\nDESPUÉS (Eliminar):')
print(f'  Datos restantes: {len(resultado_eliminar)}')
print(f'  Datos perdidos: {len(valores_con_nan) - len(resultado_eliminar)}')
print(f'  Porcentaje perdido: {(len(valores_con_nan) - len(resultado_eliminar))/len(valores_con_nan)*100:.1f}%')

### MÉTODO 2: MEDIA/MEDIANA

**Cómo funciona:** Rellena con el valor promedio (media) o central (mediana) de la columna

**Fórmula:**
- Media: `NaN = promedio de todos los valores`
- Mediana: `NaN = valor del 50%` (menos sensible a outliers)

**Cuándo usar:**
- Datos **independientes** (no temporal/espacial)
- Distribución normal

**Ventajas:**
- Rápido
- Mantiene media global

**Desventajas:**
- ❌ Crea picos artificiales (valores planos)
- ❌ Pierde variación natural

**Para datos espaciales:** ⚠️ Solo si datos muy cercanos son similares

In [ ]:
print('MÉTODO 2: MEDIA/MEDIANA')

media = valores_con_nan.mean()
mediana = valores_con_nan.median()

print(f'\nMedia de irradiancia: {media:.1f} W/m²')
print(f'Mediana de irradiancia: {mediana:.1f} W/m²')

# Aplicar
resultado_media = valores_con_nan.fillna(media)
resultado_mediana = valores_con_nan.fillna(mediana)

print(f'\nDespués de rellenar:')
print(f'  Con media: {len(resultado_media)} datos (0 faltantes)')
print(f'  Con mediana: {len(resultado_mediana)} datos (0 faltantes)')
print(f'\n✓ Todos los valores rellenados')
print(f'⚠️ Pero creará un pico artificial en {mediana:.0f} W/m²')

### MÉTODO 3: INTERPOLACIÓN LINEAL

**Cómo funciona:** Dibuja una línea entre el punto anterior y siguiente

```
Punto anterior: 100
Punto siguiente: 200
Punto faltante en el medio: (100 + 200) / 2 = 150
```

**Fórmula matemática:**
```
y = y1 + (y2 - y1) * (x - x1) / (x2 - x1)
```

**Cuándo usar:**
- **Series de tiempo** ✅ (datos meteorológicos, energía)
- Variación suave entre valores
- Datos espaciales donde vecinos son relevantes ✅

**Ventajas:**
- ✅ Mantiene forma natural
- ✅ Respeta tendencia
- ✅ Simple computacionalmente

**Desventajas:**
- Asume cambio lineal (no siempre cierto)

**Para datos espaciales:** ✅ Excelente (usa gradientes)

In [ ]:
print('MÉTODO 3: INTERPOLACIÓN LINEAL')

resultado_lineal = valores_con_nan.interpolate(method='linear')

print(f'\nDatos faltantes ANTES: {valores_con_nan.isna().sum()}')
print(f'Datos faltantes DESPUÉS: {resultado_lineal.isna().sum()}')

# Ver algunos ejemplos
print(f'\nEjemplos de interpolación:')
idx_nan = valores_con_nan[valores_con_nan.isna()].index[:3]
for i in idx_nan:
    if i > 0 and i < len(valores_con_nan)-1:
        anterior = valores_con_nan.iloc[i-1]
        siguiente = valores_con_nan.iloc[i+1]
        estimado = resultado_lineal.iloc[i]
        print(f'  Anterior: {anterior:.0f}, Siguiente: {siguiente:.0f}, Estimado: {estimado:.0f}')

print('\n✅ Mantiene forma natural (no crea picos artificiales)')
print('✅ Recomendado para series de tiempo')

### MÉTODO 4: INTERPOLACIÓN POLINOMIAL

**Cómo funciona:** Ajusta un polinomio (curva) a los datos

**Fórmula:**
```
Orden 1: y = a + b*x           (línea)
Orden 2: y = a + b*x + c*x²    (parábola)
Orden 3: y = a + b*x + c*x² + d*x³  (curva cúbica)
```

**Cuándo usar:**
- Datos con **curvas suaves** (no lineales)
- Radiación solar (forma parabólica: baja→alta→baja)
- Temperatura diaria (forma sinusoidal)

**Ventajas:**
- ✅ Ajusta curvas complejas mejor que lineal
- ✅ Más realista para algunos fenómenos

**Desventajas:**
- Puede crear oscilaciones (overfitting)
- Necesita más datos

**Para datos espaciales:** ✅ Bueno (si hay curvatura)

In [ ]:
print('MÉTODO 4: INTERPOLACIÓN POLINOMIAL')

# Orden 1 (lineal)
resultado_poly1 = valores_con_nan.interpolate(method='polynomial', order=1)

# Orden 2 (parabola)
resultado_poly2 = valores_con_nan.interpolate(method='polynomial', order=2)

# Orden 3 (cúbica)
resultado_poly3 = valores_con_nan.interpolate(method='polynomial', order=3)

print(f'Orden 1: {resultado_poly1.isna().sum()} faltantes (= lineal)')
print(f'Orden 2: {resultado_poly2.isna().sum()} faltantes (parábola)')
print(f'Orden 3: {resultado_poly3.isna().sum()} faltantes (cúbica)')

print('\n✅ Orden 2-3 captura curvas de radiación solar')
print('✅ Mejor para datos con patrones no-lineales')

### MÉTODO 5: FORWARD FILL (ffill) / BACKWARD FILL (bfill)

**Cómo funciona:** Copia el último valor conocido hacia adelante o hacia atrás

**Fórmula:**
- ffill: `NaN = valor_anterior` (repite)
- bfill: `NaN = valor_siguiente` (copia hacia atrás)

**Ejemplo:**
```
Original: [100, NaN, NaN, 200, 150]
ffill:    [100, 100, 100, 200, 150]  ← copia hacia adelante
bfill:    [100, 200, 200, 200, 150]  ← copia hacia atrás
```

**Cuándo usar:**
- Datos que **no cambian mucho** entre observaciones
- Categorías (categoría no cambia hasta siguiente cambio)
- Últimos valores en series de tiempo

**Ventajas:**
- ✅ Mantiene valores reales (no estima)
- ✅ Sencillo

**Desventajas:**
- Asume constancia (no siempre válido)

**Para datos espaciales:** ⚠️ Solo para categorías

In [ ]:
print('MÉTODO 5: FORWARD FILL Y BACKWARD FILL')

resultado_ffill = valores_con_nan.fillna(method='ffill')
resultado_bfill = valores_con_nan.fillna(method='bfill')

print(f'Forward fill (ffill): {resultado_ffill.isna().sum()} faltantes')
print(f'Backward fill (bfill): {resultado_bfill.isna().sum()} faltantes')

# Ejemplo visual
ejemplo_idx = 50:55
print(f'\nEjemplo de los primeros NaN:')
print(f'Original: {valores_con_nan.iloc[50:55].values}')
print(f'ffill:    {resultado_ffill.iloc[50:55].values}')
print(f'bfill:    {resultado_bfill.iloc[50:55].values}')

print('\n⚠️ Crea escalones (no suave)')
print('✅ Bueno para categorías, datos que no cambian')

### COMPARACIÓN: Visualizar los 5 métodos

Ahora veremos todos lado a lado en una gráfica

In [ ]:
# Seleccionar rango pequeño para claridad
rango = slice(0, 200)

fig, axes = plt.subplots(3, 2, figsize=(16, 12))
fig.suptitle('COMPARACIÓN DE 6 MÉTODOS DE IMPUTACIÓN', fontsize=14, fontweight='bold')

# 1. Original
ax = axes[0, 0]
ax.plot(valores_con_nan[rango].values, 'b-', alpha=0.6, linewidth=1, label='Datos')
ax.scatter(valores_con_nan[rango][valores_con_nan[rango].isna()].index, 
          [0]*valores_con_nan[rango].isna().sum(), color='red', s=50, marker='X', label='NaN', zorder=5)
ax.set_title('0. DATOS ORIGINALES (Con huecos)', fontweight='bold')
ax.set_ylabel('Irradiancia (W/m²)')
ax.legend()
ax.grid(True, alpha=0.3)

# 2. Lineal
ax = axes[0, 1]
ax.plot(resultado_lineal[rango].index, resultado_lineal[rango].values, 'g-', linewidth=2, marker='^', markersize=3, label='Lineal')
ax.scatter(valores_con_nan[rango][valores_con_nan[rango].isna()].index,
          resultado_lineal[rango][valores_con_nan[rango].isna()].values,
          color='darkgreen', s=100, marker='D', edgecolors='black', linewidth=1.5, label='Rellenados')
ax.set_title('1. INTERPOLACIÓN LINEAL ✅ RECOMENDADO', fontweight='bold', color='darkgreen')
ax.set_ylabel('Irradiancia (W/m²)')
ax.legend()
ax.grid(True, alpha=0.3)

# 3. Polinomial
ax = axes[1, 0]
ax.plot(resultado_poly2[rango].index, resultado_poly2[rango].values, 'purple', linewidth=2, marker='s', markersize=3, label='Polinomial orden 2')
ax.set_title('2. POLINOMIAL (orden 2) ✅ Para curvas', fontweight='bold', color='purple')
ax.set_ylabel('Irradiancia (W/m²)')
ax.legend()
ax.grid(True, alpha=0.3)

# 4. Media
ax = axes[1, 1]
ax.plot(resultado_mediana[rango].index, resultado_mediana[rango].values, 'orange', linewidth=2, marker='o', markersize=3)
ax.axhline(mediana, color='orange', linestyle='--', alpha=0.5, label=f'Mediana={mediana:.0f}')
ax.set_title('3. MEDIANA ⚠️ Crea picos planos', fontweight='bold', color='darkorange')
ax.set_ylabel('Irradiancia (W/m²)')
ax.legend()
ax.grid(True, alpha=0.3)

# 5. Forward Fill
ax = axes[2, 0]
ax.plot(resultado_ffill[rango].index, resultado_ffill[rango].values, 'brown', linewidth=2, marker='*', markersize=5, linestyle=':')
ax.set_title('4. FORWARD FILL ⚠️ Escalones', fontweight='bold', color='brown')
ax.set_ylabel('Irradiancia (W/m²)')
ax.set_xlabel('Índice')
ax.grid(True, alpha=0.3)

# 6. Eliminar (solo puntos)
ax = axes[2, 1]
ax.scatter(resultado_eliminar[rango].index, resultado_eliminar[rango].values, color='red', s=10, alpha=0.6)
ax.set_title('5. ELIMINAR ❌ Pierdes datos', fontweight='bold', color='darkred')
ax.set_ylabel('Irradiancia (W/m²)')
ax.set_xlabel('Índice')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print('✓ Comparación visual de 6 métodos')

## TABLA RESUMEN: ¿Cuándo usar cada método?

| Tipo de datos | Mejor método | Por qué | Evitar |
|---|---|---|---|
| **Series de tiempo** (meteorología) | Lineal / Spline | Respeta tendencia temporal | Mediana, Eliminar |
| **Radiación solar** | Polinomial orden 2-3 | Forma parabólica natural | Lineal puro |
| **Datos espaciales 2D** | Spline 2D | Interpolación suave | Mediana |
| **Cambios lentos** | Forward Fill | Valor anterior válido | Lineal |
| **Datos independientes** | Media/Mediana | Relación uniforme | Interpolar |
| **Con muchos outliers** | Mediana | Robusta | Media |

## PARTE 3: ESCALADO - NORMALIZAR VARIABLES

¿Por qué escalar?
- Variables con rangos distintos confunden modelos ML
- Ejemplo: irradiancia (0-900) vs viento (0-10) tienen escalas muy diferentes

### MÉTODO 1: StandardScaler (Estandarización)

**Fórmula:**
```
z = (x - media) / desv_est
```

**Resultado:** Media=0, Desv.est=1

**Cuándo usar:**
- Regresión lineal ✅
- Redes neuronales ✅
- Modelos que asumen distribución normal

**Ventajas:**
- ✅ Mantiene relación entre valores
- ✅ Preserva datos atípicos

**Desventajas:**
- No acota a rango específico (puede ser negativo/muy grande)

In [ ]:
print('MÉTODO 1: StandardScaler')

col_test = 'irradiancia_wm2'
valores = df[col_test].dropna().values.reshape(-1, 1)

print(f'\nDATA ORIGINAL:')
print(f'  Min: {valores.min():.2f}')
print(f'  Max: {valores.max():.2f}')
print(f'  Media: {valores.mean():.2f}')
print(f'  Desv.est: {valores.std():.2f}')

scaler_std = StandardScaler().fit(valores)
valores_scaled = scaler_std.transform(valores)

print(f'\nDESPUÉS DE StandardScaler:')
print(f'  Min: {valores_scaled.min():.2f}')
print(f'  Max: {valores_scaled.max():.2f}')
print(f'  Media: {valores_scaled.mean():.2f}')
print(f'  Desv.est: {valores_scaled.std():.2f}')

print('\n✅ Media → 0, Desv.est → 1')
print('✅ Para regresión y redes neuronales')

### MÉTODO 2: MinMaxScaler (Normalización [0,1])

**Fórmula:**
```
x_scaled = (x - min) / (max - min)
```

**Resultado:** Todos entre 0 y 1

**Cuándo usar:**
- Redes neuronales ✅ (necesitan [0,1])
- Cuando necesitas acotación clara

**Ventajas:**
- ✅ Acota a [0,1] (intuitivo)
- ✅ Preserva forma de distribución

**Desventajas:**
- Muy sensible a outliers (afectan max)

In [ ]:
print('MÉTODO 2: MinMaxScaler')

scaler_mm = MinMaxScaler().fit(valores)
valores_mm = scaler_mm.transform(valores)

print(f'\nDESPUÉS DE MinMaxScaler:')
print(f'  Min: {valores_mm.min():.4f}')
print(f'  Max: {valores_mm.max():.4f}')
print(f'  Media: {valores_mm.mean():.4f}')

print('\n✅ Todos los valores entre 0 y 1')
print('✅ Rango claro y acotado')
print('⚠️ Sensible a outliers')

### MÉTODO 3: RobustScaler (Usando IQR)

**Fórmula:**
```
x_scaled = (x - Q2) / (Q3 - Q1)
```
Donde Q2=mediana, Q1=25%, Q3=75%

**Resultado:** Centrado en mediana, escalado por IQR

**Cuándo usar:**
- Datos con muchos outliers ✅
- Cuando min/max no son confiables

**Ventajas:**
- ✅ Robusto a outliers (usa mediana e IQR)
- ✅ Mantiene 95% de datos en [-2, 2]

**Desventajas:**
- No acota a rango específico

In [ ]:
print('MÉTODO 3: RobustScaler')

scaler_rb = RobustScaler().fit(valores)
valores_rb = scaler_rb.transform(valores)

print(f'\nDESPUÉS DE RobustScaler:')
print(f'  Min: {valores_rb.min():.2f}')
print(f'  Max: {valores_rb.max():.2f}')
print(f'  Mediana: {np.median(valores_rb):.2f}')
print(f'  IQR: {np.percentile(valores_rb, 75) - np.percentile(valores_rb, 25):.2f}')

print('\n✅ Usa mediana e IQR (resistente a extremos)')
print('✅ Mejor para datos con outliers')

### MÉTODO 4: LogScaler

**Fórmula:**
```
x_scaled = log(1 + x)
```

**Resultado:** Comprime valores grandes

**Cuándo usar:**
- Distribución log-normal ✅
- Cuando datos tienen muchos pequeños y pocos grandes
- Ejemplo: ingresos, tamaño de ciudades

**Ventajas:**
- ✅ Normaliza distribuciones sesgadas
- ✅ Comprime outliers

**Desventajas:**
- No es lineal (cambia relaciones)

In [ ]:
print('MÉTODO 4: LogScaler')

valores_log = np.log1p(valores)  # log(1+x) para evitar log(0)

print(f'\nDESPUÉS DE LogScaler:')
print(f'  Min: {valores_log.min():.2f}')
print(f'  Max: {valores_log.max():.2f}')
print(f'  Media: {valores_log.mean():.2f}')

print('\n✅ Comprime valores grandes')
print('✅ Normaliza distribuciones log-normales')

In [ ]:
# VISUALIZAR TODOS LOS ESCALADORES
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
fig.suptitle('COMPARACIÓN DE 4 MÉTODOS DE ESCALADO', fontsize=14, fontweight='bold')

# Original
ax = axes[0, 0]
ax.hist(valores, bins=30, color='blue', alpha=0.7, edgecolor='black')
ax.set_title('0. ORIGINAL\n[0, 896], μ=213', fontweight='bold')
ax.set_ylabel('Frecuencia')
ax.grid(True, alpha=0.3)

# Standard
ax = axes[0, 1]
ax.hist(valores_scaled, bins=30, color='green', alpha=0.7, edgecolor='black')
ax.set_title('1. STANDARD\nμ≈0, σ≈1\nPara: Regresión, NN', fontweight='bold', color='darkgreen')
ax.set_ylabel('Frecuencia')
ax.grid(True, alpha=0.3)

# MinMax
ax = axes[0, 2]
ax.hist(valores_mm, bins=30, color='orange', alpha=0.7, edgecolor='black')
ax.set_title('2. MINMAX [0,1]\nAcota [0,1]\nPara: Redes NN', fontweight='bold', color='darkorange')
ax.set_ylabel('Frecuencia')
ax.grid(True, alpha=0.3)

# Robust
ax = axes[1, 0]
ax.hist(valores_rb, bins=30, color='purple', alpha=0.7, edgecolor='black')
ax.set_title('3. ROBUST\nResistente outliers\nPara: Datos extremos', fontweight='bold', color='purple')
ax.set_ylabel('Frecuencia')
ax.grid(True, alpha=0.3)

# Log
ax = axes[1, 1]
ax.hist(valores_log, bins=30, color='brown', alpha=0.7, edgecolor='black')
ax.set_title('4. LOG\nComprime grandes\nPara: Log-normal', fontweight='bold', color='brown')
ax.set_ylabel('Frecuencia')
ax.grid(True, alpha=0.3)

# Comparación
ax = axes[1, 2]
ax.boxplot([valores.flatten()/100, valores_scaled.flatten(), valores_mm.flatten(), valores_rb.flatten()],
          labels=['Original/100', 'Standard', 'MinMax', 'Robust'])
ax.set_title('Comparación de distribuciones', fontweight='bold')
ax.set_ylabel('Valor')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print('✓ Todos los escaladores comparados visualmente')

## TABLA FINAL: Recomendaciones por modelo

| Modelo ML | Escalador | Por qué |
|---|---|---|
| **Regresión lineal** | StandardScaler | Asume distribución normal |
| **Redes neuronales** | MinMaxScaler | Necesita [0,1] |
| **SVM** | StandardScaler | Sensible a escala |
| **KNN** | MinMaxScaler | Distancia euclideana |
| **Random Forest** | Ninguno | Invariante a escala |
| **Con outliers** | RobustScaler | Robusto |
| **Datos log-normal** | LogScaler | Normaliza distribución |

## RESUMEN FINAL

✅ **Imputación:** Interpola lineal para series de tiempo

✅ **Escalado:** StandardScaler para regresión, MinMaxScaler para redes neuronales

✅ **Siempre:** Ajusta escalador SOLO con entrenamiento (evita fuga de datos)